In [0]:
-- ============================================================
-- PROYECTO INTEGRADOR ELECTROCASA
-- Notebook: 00_setup
-- Objetivo:
--   - Verificar el entorno Databricks
--   - Validar catalogos y schemas
--   - Documentar la configuracion base del Lakehouse
-- ============================================================

SELECT
    current_catalog() AS catalogo_actual,
    current_schema() AS schema_actual,
    current_user() AS usuario_actual;

In [0]:
SHOW SCHEMAS IN electrocasa_dev;

In [0]:
SHOW TABLES IN electrocasa_dev.bronze;

In [0]:
SHOW TABLES IN electrocasa_dev.silver;

In [0]:
SHOW TABLES IN electrocasa_dev.gold;

In [0]:
SHOW TABLES IN electrocasa_dev.audit;

In [0]:
SHOW CATALOGS LIKE 'electrocasa_prod';

In [0]:
-- ============================================================
-- ELECTROCASA - PREPARACION ENTORNO PROD
-- ============================================================

CREATE CATALOG IF NOT EXISTS electrocasa_prod
COMMENT 'Catalogo de produccion del proyecto ElectroCasa';

CREATE SCHEMA IF NOT EXISTS electrocasa_prod.bronze
COMMENT 'Capa Bronze de produccion';

CREATE SCHEMA IF NOT EXISTS electrocasa_prod.silver
COMMENT 'Capa Silver de produccion';

CREATE SCHEMA IF NOT EXISTS electrocasa_prod.gold
COMMENT 'Capa Gold de produccion';

CREATE SCHEMA IF NOT EXISTS electrocasa_prod.audit
COMMENT 'Cuarentena y auditoria de produccion';

CREATE VOLUME IF NOT EXISTS electrocasa_prod.bronze.landing
COMMENT 'Archivos fuente de ElectroCasa para produccion';

In [0]:
SHOW SCHEMAS IN electrocasa_prod;

In [0]:
SHOW VOLUMES IN electrocasa_prod.bronze;

In [0]:
%python

# ============================================================
# ELECTROCASA - COPIA DE FUENTES DEV -> PROD
# ============================================================

archivos = [
    (
        "/Volumes/electrocasa_dev/bronze/landing/productos/catalogo_productos.json",
        "/Volumes/electrocasa_prod/bronze/landing/productos/catalogo_productos.json"
    ),
    (
        "/Volumes/electrocasa_dev/bronze/landing/ventas/ventas_sucursales.csv",
        "/Volumes/electrocasa_prod/bronze/landing/ventas/ventas_sucursales.csv"
    ),
    (
        "/Volumes/electrocasa_dev/bronze/landing/devoluciones/devoluciones.csv",
        "/Volumes/electrocasa_prod/bronze/landing/devoluciones/devoluciones.csv"
    ),
    (
        "/Volumes/electrocasa_dev/bronze/landing/empleados/empleados_rrhh.csv",
        "/Volumes/electrocasa_prod/bronze/landing/empleados/empleados_rrhh.csv"
    ),
    (
        "/Volumes/electrocasa_dev/bronze/landing/resenas/resenas_clientes.json",
        "/Volumes/electrocasa_prod/bronze/landing/resenas/resenas_clientes.json"
    )
]

for origen, destino in archivos:
    carpeta_destino = destino.rsplit("/", 1)[0]

    dbutils.fs.mkdirs(carpeta_destino)
    resultado = dbutils.fs.cp(origen, destino)

    if resultado:
        print(f"OK: {destino}")
    else:
        print(f"ERROR: {destino}")